<a href="https://colab.research.google.com/github/huy-V0/ai_research/blob/main/notebooks/setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Tue Jul 28 21:00:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             27W /   70W |    3107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/persona_rqa/vectors
!mkdir -p /content/drive/MyDrive/persona_rqa/results
!mkdir -p /content/drive/MyDrive/persona_rqa/cache
!df -h /content/drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filesystem      Size  Used Avail Use% Mounted on
drive            15G   15G  427M  98% /content/drive


In [ ]:
!pip install -q transformers==4.51.0 accelerate datasets

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name, torch_dtype=torch.float16, device_map="cuda"
)
model.eval()
print(model.config.num_hidden_layers, model.config.hidden_size)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

28 1536


In [ ]:
def get_resid(prompts, layer):
    outs = []
    for p in prompts:
        msgs = [{"role": "user", "content": p}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tok(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            h = model(**ids, output_hidden_states=True).hidden_states
        outs.append(h[layer][0, -1, :].float().cpu())
    return torch.stack(outs)

a = get_resid(["How do I bake bread?", "Explain gravity."], 14)
print(a.shape, a.dtype)

torch.Size([2, 1536]) torch.float32
